# Compare robot controllers

Compares robot controllers verified by Verify_RCCO. It reads the `errors.csv` of each verify run listed in an experiment of type robot_controller_compare (the `all` row: the per-field MSE over all testing demonstrations) and charts them side by side. It recomputes nothing; Verify_RCCO is the single source of the numbers.

See robot_controller/DESIGN-BehaviorCloningFlow.md.

In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import csv
import pathlib
from pprint import pprint
import matplotlib.pyplot as plt
# fixing the fact that it defaults to Type 3 fonts
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import numpy as np

In [ ]:
# *** Initialize the variables with default values
# *** This cell is tagged as parameters
# *** If papermill is used, some of the values will be overwritten

creation_style = "exist-ok"
experiment = "robot_controller_compare"
run = "compare_sample"

# If not None, set an external experiment path
expruns_path = None
# If not None, set an external results path
results_path = None

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path).expanduser()
    assert expruns_path.exists()
    Config().set_exprun_path(expruns_path)
if results_path:
    results_path = pathlib.Path(results_path).expanduser()
    assert results_path.exists()
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
pprint(exp)

## Collect the errors of the verify runs

In [ ]:
errors = {}
for verify_run in exp["verify_runs"]:
    exp_verify = Config().get_experiment(exp["verify_experiment"], verify_run)
    with open(pathlib.Path(exp_verify.data_dir(), "errors.csv")) as f:
        rows = {row["demonstration"]: row for row in csv.DictReader(f)}
    errors[verify_run] = rows["all"]

fields = [name for name in next(iter(errors.values())) if name != "demonstration"]
comparison_file = pathlib.Path(exp.data_dir(), "comparison.csv")
with open(comparison_file, "w") as f:
    f.write("verify_run,label," + ",".join(fields) + "\n")
    for (verify_run, row), label in zip(errors.items(), exp["labels"], strict=True):
        f.write(f"{verify_run},{label}," + ",".join(row[field] for field in fields) + "\n")
print(comparison_file.read_text())

## Per-field error of each controller

In [ ]:
x = np.arange(len(fields))
width = 0.8 / len(errors)
fig, ax = plt.subplots(figsize=(10, 4))
for i, (label, row) in enumerate(zip(exp["labels"], errors.values(), strict=True)):
    ax.bar(x + i * width, [float(row[field]) for field in fields], width,
           label=label)
ax.set_xticks(x + width * (len(errors) - 1) / 2)
ax.set_xticklabels(fields)
ax.set_ylabel("MSE (normalized)")
ax.set_title("Teacher-forced error on the testing demonstrations")
ax.legend(title="Controller")
for suffix in [".pdf", ".png"]:
    plt.savefig(pathlib.Path(exp.data_dir(), "comparison" + suffix), bbox_inches="tight")
plt.show()

In [ ]:
exp.done()